# CoWork Enterprise Demo — Control the Cost
## Notebook 04 — Control the Cost (Phase 4)

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

> ⚠️ _Generated from `build_notebooks.py` — edit the builder and re-run. Direct edits to this notebook are overwritten._

---

### What This Notebook Does:

1. 🛠️ **Observes** AI spend with Account Usage views
2. 🛠️ **Attributes** every credit to a cost centre with tags (agent + warehouse)
3. 🛠️ **Controls** spend with a warehouse timeout, a per-agent budget + alert, and a per-user quota

---

### Why AI Cost Needs Its Own Discipline:

🔹 AI credits can be harder to predict than warehouse compute, and they come from **two meters**: warehouse compute (the SQL Cortex Analyst generates) **and** serverless LLM reasoning (billed separately). Manage both from day one with a simple framework: **Observe → Attribute → Control**.

📌 **Attribution rule:** a request that starts in CoWork and invokes an agent is billed to **CoWork**, not to the agent object. To see the full picture, run *both* a CoWork-scoped budget and per-agent budgets.

> **Documentation:** [AI cost management](https://docs.snowflake.com/en/user-guide/snowflake-cortex/governance-and-availability/ai-cost-management-and-governance)

---

### Estimated Time: **15–20 minutes**

### Prerequisites:
- Notebook 03 complete (agent created and tested)

In [ ]:
%%sql -r set_context
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
USE DATABASE COWORK_ENTERPRISE_DEMO;
USE SCHEMA AGENTS;
USE WAREHOUSE COWORK_ENTERPRISE_DEMO_WH;


## Step 1: Observe — Know What You're Spending

🛠️ Before you can control cost you need visibility. `CORTEX_AGENT_USAGE_HISTORY` shows per-agent token spend; `METERING_DAILY_HISTORY` shows the account-wide AI credit trend.

In [ ]:
%%sql -r observe_agent
SELECT START_TIME, USER_NAME, AGENT_NAME, TOKENS, TOKEN_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
WHERE AGENT_NAME = 'DEMO_AGENT'
ORDER BY START_TIME DESC
LIMIT 20;


In [ ]:
%%sql -r observe_account
SELECT USAGE_DATE, SERVICE_TYPE, SUM(CREDITS_USED) AS TOTAL_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE = 'AI_SERVICES'
  AND USAGE_DATE >= DATEADD('day', -7, CURRENT_DATE())
GROUP BY USAGE_DATE, SERVICE_TYPE
ORDER BY USAGE_DATE DESC;


## Step 2: Attribute — Know Who Is Spending What

🛠️ Tag the agent **and** its warehouse with a cost centre so one tag-based budget or quota captures everything for that team.

📌 **Tag before you budget** — retroactive tagging only captures *future* spend.

In [ ]:
%%sql -r create_tag
CREATE TAG IF NOT EXISTS COWORK_ENTERPRISE_DEMO.AGENTS.COST_CENTER
  ALLOWED_VALUES 'demo', 'finance', 'shared'
  COMMENT = 'Cost-centre attribution for CoWork / agent spend';


In [ ]:
%%sql -r tag_agent
ALTER AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT SET TAG COWORK_ENTERPRISE_DEMO.AGENTS.COST_CENTER = 'demo';


In [ ]:
%%sql -r tag_warehouse
ALTER WAREHOUSE COWORK_ENTERPRISE_DEMO_WH SET TAG COWORK_ENTERPRISE_DEMO.AGENTS.COST_CENTER = 'demo';


> **Tag users too (reference).** Per-user chargeback needs the user object tagged; we don't tag a real user on a shared account:
```sql
ALTER USER <business_user> SET TAG COWORK_ENTERPRISE_DEMO.AGENTS.COST_CENTER = 'demo';
```

## Step 3: Control — Warehouse Guardrail
🛠️ Cap runaway SQL with a **statement timeout** on the agent's warehouse. In production, use a dedicated warehouse per agent category (e.g. `COWORK_FINANCE_WH`), sized XS and upsized only if query performance requires it. (Cortex LLM reasoning is serverless and billed separately from this warehouse compute - budget both.)

In [ ]:
%%sql -r harden_warehouse
ALTER WAREHOUSE COWORK_ENTERPRISE_DEMO_WH SET STATEMENT_TIMEOUT_IN_SECONDS = 120;


## Step 4: Control — Per-Agent Budget

🛠️ A **budget** tracks credits against a monthly limit and alerts (or acts) at a threshold. Enforcement is periodic, so alert well below the hard stop.

In [ ]:
%%sql -r create_budget
CREATE OR REPLACE SNOWFLAKE.CORE.BUDGET COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_BUDGET();
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_BUDGET!SET_SPENDING_LIMIT(50);


In [ ]:
%%sql -r budget_threshold
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_BUDGET!SET_NOTIFICATION_THRESHOLD(75);


> **Email alerts (reference).** Email needs verified recipients (or a notification integration); we don't send mail from a shared account:
```sql
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_BUDGET!SET_EMAIL_NOTIFICATIONS('costadmin@yourco.com');
-- raise the threshold for a second, higher alert:
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_BUDGET!SET_NOTIFICATION_THRESHOLD(90);
```

## Step 5: Control — Per-User Quota
🛠️ A **quota** is a first-class object that caps AI credits **per user** and can auto-block at the limit. We create it in **monitoring mode** (no enforcement, no notifications) so it is safe on a shared account. Scope it to CoWork **and** the agent - a CoWork-initiated request is billed to CoWork, not the agent (the attribution rule from the intro), so cover both.

In [ ]:
%%sql -r create_quota
USE ROLE ACCOUNTADMIN;
CREATE SNOWFLAKE.CORE.QUOTA COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA();
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA!ADD_SHARED_RESOURCE('SNOWFLAKE INTELLIGENCE');
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA!ADD_SHARED_RESOURCE('CORTEX AGENT', (SELECT SYSTEM$REFERENCE('CORTEX AGENT', 'COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT')));
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA!SET_PER_USER_LIMIT(50);
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA!SET_PER_USER_LIMIT(5, 'DAILY');
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
SELECT 'Quota created (monitoring only)' AS STATUS;


> **Enforce (reference).** In production, turn on automatic blocking and a proactive alert:
```sql
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA!SET_BLOCK_ENFORCEMENT_ENABLED(TRUE);
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA!ADD_NOTIFICATION_THRESHOLD(80, 'PROJECTED', TRUE);
```
> Note: the legacy `ALTER USER ... SET MONTHLY_CORTEX_AI_CREDITS_QUOTA` form is **not** current syntax - the quota object above is the supported mechanism.

## Step 6: Budget Models — Resource-Level vs Shared Resource

🔹 Beyond the **per-agent** budget above, Snowflake CoWork supports two budget *scopes* — pick based on your governance need:

| Model | Scope | Tag what | Impact |
|---|---|---|---|
| **Resource-Level** | The entire CoWork (SI) object | the **SI object** | **all** users of the object |
| **Shared Resource** | A subset of users (a team) | **individual users** | just that team/group |

🔹 A CoWork-initiated request bills to the **CoWork object**, so an SI-scoped budget is how you cap total CoWork spend; a shared-resource budget gives each team its own cap on the *same* object. When a user is covered by several budgets, the **first threshold reached** stops them.

### 🛠️ Shared Resource budget (team-scoped) — runnable
This is the safe, executable model: it tags **users** (not the shared object). We cap a team at 200 credits/month on CoWork, tracked only for users carrying our `cost_centre` tag. Tagging real users is left as reference (no demo users on a shared account).

In [ ]:
%%sql -r create_team_budget
CREATE OR REPLACE SNOWFLAKE.CORE.BUDGET COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_TEAM_BUDGET();
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_TEAM_BUDGET!SET_SPENDING_LIMIT(200);
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_TEAM_BUDGET!ADD_SHARED_RESOURCE('SNOWFLAKE INTELLIGENCE');
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_TEAM_BUDGET!SET_USER_TAGS(
  [ [(SELECT SYSTEM$REFERENCE('TAG', 'COWORK_ENTERPRISE_DEMO.AGENTS.COST_CENTER', 'SESSION', 'APPLYBUDGET')), 'demo'] ],
  'UNION');
-- Reference: apply the tag to the team's users so the budget tracks them:
-- ALTER USER <business_user> SET TAG COWORK_ENTERPRISE_DEMO.AGENTS.COST_CENTER = 'demo';


### 📌 Resource-Level budget (SI object, all users) — reference only
This caps the **entire shared CoWork object across every team**, so we do **not** run it here — tagging the shared singleton would affect other teams on this account.

```sql
CREATE OR REPLACE SNOWFLAKE.CORE.BUDGET org_si_budget();
CALL org_si_budget!SET_SPENDING_LIMIT(10000);
-- Tag the SHARED object (org-level cap):
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT SET TAG COWORK_ENTERPRISE_DEMO.AGENTS.COST_CENTER = 'shared';
CALL org_si_budget!SET_RESOURCE_TAGS(
  [ [(SELECT SYSTEM$REFERENCE('TAG', 'COWORK_ENTERPRISE_DEMO.AGENTS.COST_CENTER', 'SESSION', 'APPLYBUDGET')), 'shared'] ],
  'UNION');
```

### 📌 Threshold actions with stored procedures — reference only
Budgets can **run a stored procedure at a threshold** (alert at 80%, block at 100%). Blocking works by revoking a **dedicated per-team role**, and the proc must be granted to the `SNOWFLAKE` application. Enforcement is periodic (up to ~8h, or ~2h with a low-latency budget), so alert well below the hard stop. Shown for reference — it needs a dedicated role and an app grant.

```sql
CREATE OR REPLACE PROCEDURE COWORK_ENTERPRISE_DEMO.AGENTS.SP_REVOKE_TEAM(dept STRING)
  RETURNS STRING LANGUAGE SQL AS
BEGIN
  EXECUTE IMMEDIATE 'REVOKE ROLE si_' || :dept || '_role FROM ROLE ' || :dept || '_role';
  RETURN 'Access revoked for ' || :dept;
END;
GRANT USAGE ON PROCEDURE COWORK_ENTERPRISE_DEMO.AGENTS.SP_REVOKE_TEAM(STRING) TO APPLICATION SNOWFLAKE;

-- Block the team at 100% of its shared budget:
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_TEAM_BUDGET!ADD_CUSTOM_ACTION(
  SYSTEM$REFERENCE('PROCEDURE', 'COWORK_ENTERPRISE_DEMO.AGENTS.SP_REVOKE_TEAM(string)'),
  ARRAY_CONSTRUCT('demo'), 'ACTUAL', 100);
```
> The same `ADD_CUSTOM_ACTION` pattern with `'PROJECTED'` fires on *forecast* spend, and thresholds up to 1000% let you script reinstatement for peak periods.

## Step 7: Verify

In [ ]:
%%sql -r verify_tag
SHOW TAGS IN SCHEMA COWORK_ENTERPRISE_DEMO.AGENTS;


In [ ]:
%%sql -r verify_budget
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_BUDGET!GET_SPENDING_LIMIT();


In [ ]:
%%sql -r verify_team_budget
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_TEAM_BUDGET!GET_BUDGET_SCOPE();


In [ ]:
%%sql -r verify_quota
CALL COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA!GET_CONFIG();


## ✅ Notebook 04 Complete

### What You Just Built:
- Cost-centre tag on the agent and warehouse
- Warehouse statement timeout (runaway-query guardrail)
- Per-agent resource budget with a 75% alert threshold
- A team-scoped shared resource budget on CoWork (Resource-Level + stored-proc threshold actions shown as reference)
- Per-user quota (monitoring mode) scoped to CoWork + the agent

---

### Key Points:
- AI spend comes from two meters — warehouse compute and serverless LLM reasoning. Budget both.
- CoWork-initiated spend attributes to CoWork, not the agent — so run both budget scopes.
- Tag before you budget; quotas cap individual users where budgets cap a team/agent.

---

### Next:

Continue to **Notebook 05 — Evaluate, Go Live & Operate**: score the agent with Agent GPA, publish it to CoWork under governance, and brand the interface.